In [1]:
import re
import pickle
from langchain_community.retrievers import BM25Retriever
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader


/tmp/ipykernel_147908/1262777416.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
/home/yan/miniconda3/envs/LocalAgent_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load trial documents

Load each Markdown file as one document and attach its NCT ID as metadata.

In [2]:
md_dir = Path("data/trec_pm2019/trial_markdown")

loader = DirectoryLoader(
    str(md_dir),
    glob="NCT*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = loader.load()
documents.sort(key=lambda doc: doc.metadata["source"])

for document in documents:
    document.metadata["nct_id"] = Path(document.metadata["source"]).stem

print("Documents:", len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:500])

100%|██████████| 8567/8567 [00:03<00:00, 2307.21it/s]

Documents: 8567
{'source': 'data/trec_pm2019/trial_markdown/NCT00000106.md', 'nct_id': 'NCT00000106'}
## Title

41.8 Degree Centigrade Whole Body Hyperthermia for the Treatment of Rheumatoid Diseases

## NCT ID

NCT00000106

## Conditions

Rheumatic Diseases

## Interventions

Whole body hyperthermia unit

## Brief Summary

Recently a non-toxic system for whole body hyperthermia (WBH) used at the University of Wisconsin has been shown to induce soluble tumor necrosis factor-receptor (sTNF-R) I and II when patients are heated systemically to 41.8C for 60 minutes. This observation might provide a 


### Build BM25

Use the same lowercase tokenization for documents and queries.

In [3]:
bm25_retriever = BM25Retriever.from_documents(
    documents,
    preprocess_func=lambda text: re.findall(r"[a-z0-9]+", text.lower()),
    k=10,
)

print("Indexed documents:", len(bm25_retriever.docs))
bm25 = bm25_retriever.vectorizer
print("Document count:", bm25.corpus_size)
print("Average document length:", bm25.avgdl)
print("Document length:", bm25.doc_len[0])
print("Document word frequencies:", bm25.doc_freqs[0])

Indexed documents: 8567
Document count: 8567
Average document length: 811.4962063732929
Document length: 431
Document word frequencies: {'title': 1, '41': 2, '8': 1, 'degree': 2, 'centigrade': 1, 'whole': 3, 'body': 3, 'hyperthermia': 3, 'for': 11, 'the': 20, 'treatment': 2, 'of': 22, 'rheumatoid': 4, 'diseases': 3, 'nct': 1, 'id': 1, 'nct00000106': 1, 'conditions': 1, 'rheumatic': 1, 'interventions': 1, 'unit': 1, 'brief': 1, 'summary': 1, 'recently': 1, 'a': 10, 'non': 1, 'toxic': 1, 'system': 1, 'wbh': 3, 'used': 1, 'at': 6, 'university': 1, 'wisconsin': 1, 'has': 2, 'been': 2, 'shown': 1, 'to': 8, 'induce': 2, 'soluble': 1, 'tumor': 1, 'necrosis': 1, 'factor': 1, 'receptor': 1, 'stnf': 2, 'r': 2, 'i': 1, 'and': 9, 'ii': 2, 'when': 1, 'patients': 8, 'are': 5, 'heated': 1, 'systemically': 1, '8c': 1, '60': 5, 'minutes': 2, 'this': 3, 'observation': 1, 'might': 1, 'provide': 1, 'biological': 2, 'basis': 2, 'therapeutic': 1, 'application': 1, 'which': 2, 'there': 1, 'is': 4, 'positive'

### Retrieve candidate trials

In [4]:
query = "melanoma BRAF V600E"

results = bm25_retriever.invoke(query)

for rank, document in enumerate(results, start=1):
    print("Rank:", rank)
    print("NCT ID:", document.metadata["nct_id"])
    print(document.page_content[:500])
    print("-" * 80)

Rank: 1
NCT ID: NCT01136967
## Title

An Open-Label, 2-Cohort, Multicenter, Study of Lenvatinib in Previously Treated Subjects With Unresectable Stage III or Stage IV Melanoma

## NCT ID

NCT01136967

## Conditions

Unresectable Stage III, Stage IV Melanoma

## Interventions

Lenvatinib

## Brief Summary

The purpose of this study is to assess the objective response rate of lenvatinib in previously treated participants with American Joint Committee on Cancer (AJCC) unresectable Stage III or Stage IV melanoma and disease pr
--------------------------------------------------------------------------------
Rank: 2
NCT ID: NCT00405587
## Title

Safety Study of PLX4032 in Patients With Solid Tumors

## NCT ID

NCT00405587

## Conditions

Malignant Melanoma, Colorectal Carcinoma

## Interventions

PLX4032

## Brief Summary

The primary objective of this FIH study is to assess the safety and pharmacokinetics of PLX4032 in patients with solid tumors. The secondary objective is to assess the pha

### Save the index

Save BM25 statistics, ordered document metadata and tokenization settings.

In [6]:
index_path = Path("../bm25/bm25_index.pkl")

index_data = {
    "vectorizer": bm25_retriever.vectorizer,
    "metadatas": [
        document.metadata
        for document in bm25_retriever.docs
    ],
    "token_pattern": r"[a-z0-9]+",
    "lowercase": True,
}

with index_path.open("wb") as file:
    pickle.dump(index_data, file)

### Load the saved index

Restore the retriever with the saved statistics and source documents in their original order.

In [7]:
import pickle
import re
from langchain_community.document_loaders import TextLoader
from langchain_community.retrievers import BM25Retriever

with index_path.open("rb") as file:
    saved_index = pickle.load(file)

documents = []

for metadata in saved_index["metadatas"]:
    document = TextLoader(
        metadata["source"],
        encoding="utf-8",
    ).load()[0]
    document.metadata.update(metadata)
    documents.append(document)

restored_retriever = BM25Retriever(
    vectorizer=saved_index["vectorizer"],
    docs=documents,
    preprocess_func=lambda text: re.findall(
        saved_index["token_pattern"],
        text.lower() if saved_index["lowercase"] else text,
    ),
    k=10,
)

### Query the restored retriever

In [8]:
results = restored_retriever.invoke("melanoma BRAF V600E")

for rank, document in enumerate(results, start=1):
    print(rank, document.metadata["nct_id"])

1 NCT01136967
2 NCT00405587
3 NCT01928940
4 NCT03224208
5 NCT01264380
6 NCT01584648
7 NCT01597908
8 NCT01942993
9 NCT02416232
10 NCT01543113
